<!-- field-paper-role-banner -->
> 🟩 **정답·해설본** — 같은 셀 위치의 실행 가능한 참고 구현입니다. 복사하기 전에 수식과 tensor shape를 설명하세요.

# 컴퓨터 비전 · 08. Vision Transformer (ViT)

## An Image is Worth 16x16 Words: Transformers for Image Recognition at Scale

- **원 논문:** [An Image is Worth 16x16 Words: Transformers for Image Recognition at Scale](https://arxiv.org/abs/2010.11929)
- **저자 / 발표:** Alexey Dosovitskiy et al. · ICLR 2021 (2020)
- **난이도 / 예상 시간:** 중급 · 약 120분
- **선수 지식:** Python, PyTorch tensor, 합성곱, 역전파의 기초
- **로컬 학습 데이터:** `data/field_curriculum/vision_shapes.npz`

이 자료는 핵심 메커니즘을 확인하는 **하드웨어 인식 교육용 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하고, NumPy·환경 simulation 같은
CPU 연산은 그대로 유지합니다. 원 논문의 전체 데이터·모델·학습 예산·최종 benchmark를
재현했다는 뜻은 아닙니다.

## 재현 범위

**보존하는 핵심:** patch embedding, class token, position embedding, pre-LN encoder를 재현한다.

**축소하거나 생략한 부분:** 224×224/patch16, 대규모 사전학습 대신 16×16/patch4와 1 encoder layer를 쓴다.

완료 기준은 정답과 문자가 같은 코드가 아니라, TODO 구현·불변식 assertion·작은 metric 또는
시각화·마지막 해석을 모두 마치는 것입니다.

## 원 논문 ↔ 실습 지도

        절·수식·그림 번호는 위 primary source를 기준으로 합니다. 판본에 따라 페이지가 달라질 수
        있으므로 절 제목과 식 번호를 함께 확인하세요.

        | 원 논문의 위치 | 대응 코드·실험 | 확인할 증거 |
        |---|---|---|
        | §3.1 Fig. 1: patches as tokens | Task 1 patchify | 16 patches |
| §3.1 Eq. (1–4): class/position embedding and LN/MSA/MLP residual | Task 2 explicit Q/K/V pre-LN encoder | 17 tokens and [B,H,17,17] attention weights |
| §3.2: fine-tuning | Task 3 | loss/accuracy |

## 핵심 아이디어와 수식

$$\mathbf{z}_0=[\mathbf{x}_{class};\mathbf{x}_p^1E;\cdots;\mathbf{x}_p^NE]+E_{pos}$$

**기호 해설:** x_p는 펼친 patch, E는 projection, x_class는 학습 token, E_pos는 위치 embedding이다.

**코드 대응:** Task 1이 patchify/config, Task 2가 class token과 Q/K/V projection, head 분할, pre-LN residual encoder를 구현한다.

**입력과 출력 shape:** 입력 [B,1,16,16] → patches [B,16,16] → tokens [B,17,24] → logits [B,4].

**포트폴리오 구현 범위:** 대규모 사전학습과 hybrid CNN은 제외하지만 tokenization과 encoder 식은 유지한다.

아래 코드는 결과만 내는 라이브러리 호출로 핵심을 감추지 않는다. config, 논문 고유
연산, 모델의 `forward`, 목적함수, `train_step`, 평가 함수를 순서대로 직접 작성한다.

In [ ]:
from dataclasses import dataclass
from pathlib import Path
import itertools
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from llm_engineering_lab.acceleration import get_accelerator

torch.manual_seed(20260814)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device


def locate(relative_path):
    for base in (Path.cwd(), *Path.cwd().parents):
        candidate = base / relative_path
        if candidate.exists():
            return candidate
    raise FileNotFoundError(relative_path)


dataset_path = locate("data/field_curriculum/vision_shapes.npz")
raw = np.load(dataset_path)
images = torch.from_numpy(raw["images"]).float()
labels = torch.from_numpy(raw["labels"]).long()
masks = torch.from_numpy(raw["masks"]).float()
boxes = torch.from_numpy(raw["boxes"]).float()
train_idx = torch.from_numpy(raw["train_idx"][:128]).long()
test_idx = torch.from_numpy(raw["test_idx"][:64]).long()
images, labels, masks, boxes = ACCELERATOR.move(
    images,
    labels,
    masks,
    boxes,
)
train_idx, test_idx = ACCELERATOR.move(train_idx, test_idx)
class_names = raw["class_names"].tolist()
x_train = images[train_idx]
y_train = labels[train_idx]
assert images.shape == (480, 1, 16, 16)
assert masks.shape == (480, 16, 16)
assert boxes.shape == (480, 4)
print(ACCELERATOR.summary())
print(dataset_path.name, images.shape, class_names)

### 구현 단계 1 — 원문 근거를 코드로 옮기기

- **논문의 어느 부분인가:** §3.1 Fig. 1: patches as tokens
- **노트북에서 구현할 부분:** Task 1 patchify
- **구현 이유:** 논문의 계산 경로를 작은 tensor로 보존하면서 각 중간값을
  assertion으로 관찰하기 위해서다.
- **완료 증거:** 16 patches

먼저 입력 shape와 기대 출력 shape를 종이에 적은 뒤 아래 TODO를 구현한다.

In [ ]:
@dataclass
class ViTConfig:
    image_size: int = 16
    patch_size: int = 4
    embedding_dim: int = 24
    heads: int = 4
    mlp_dim: int = 48
    num_classes: int = 4
    learning_rate: float = 0.012
    steps: int = 15

    @property
    def patch_count(self):
        return (self.image_size // self.patch_size) ** 2


def patchify(images, patch_size):
    patches = images.unfold(2, patch_size, patch_size)
    patches = patches.unfold(3, patch_size, patch_size)
    patches = patches.permute(0, 2, 3, 1, 4, 5)
    return patches.reshape(len(images), -1, patch_size * patch_size)


config = ViTConfig()
patches = patchify(images[:2], config.patch_size)
assert patches.shape == (2, 16, 16)

### 구현 단계 2 — 원문 근거를 코드로 옮기기

- **논문의 어느 부분인가:** §3.1 Eq. (1–4): class/position embedding and LN/MSA/MLP residual
- **노트북에서 구현할 부분:** Task 2 explicit Q/K/V pre-LN encoder
- **구현 이유:** 논문의 계산 경로를 작은 tensor로 보존하면서 각 중간값을
  assertion으로 관찰하기 위해서다.
- **완료 증거:** 17 tokens and [B,H,17,17] attention weights

먼저 입력 shape와 기대 출력 shape를 종이에 적은 뒤 아래 TODO를 구현한다.

In [ ]:
class ExplicitMultiHeadAttention(nn.Module):
    """Q/K/V projection부터 scaled dot-product attention까지 계산한다."""

    def __init__(self, model_dim, heads):
        super().__init__()
        if model_dim % heads != 0:
            raise ValueError("model_dim must be divisible by heads")
        self.heads = heads
        self.head_dim = model_dim // heads
        self.query_projection = nn.Linear(model_dim, model_dim)
        self.key_projection = nn.Linear(model_dim, model_dim)
        self.value_projection = nn.Linear(model_dim, model_dim)
        self.output_projection = nn.Linear(model_dim, model_dim)

    def split_heads(self, tensor):
        batch, length, _ = tensor.shape
        tensor = tensor.reshape(batch, length, self.heads, self.head_dim)
        return tensor.transpose(1, 2)

    def merge_heads(self, tensor):
        batch, _, length, _ = tensor.shape
        tensor = tensor.transpose(1, 2).contiguous()
        return tensor.reshape(batch, length, self.heads * self.head_dim)

    def forward(self, query, key, value, attention_mask=None):
        queries = self.split_heads(self.query_projection(query))
        keys = self.split_heads(self.key_projection(key))
        values = self.split_heads(self.value_projection(value))
        scale = self.head_dim**-0.5
        scores = torch.matmul(queries, keys.transpose(-2, -1)) * scale
        if attention_mask is not None:
            scores = scores.masked_fill(attention_mask, float("-inf"))
        weights = scores.softmax(dim=-1)
        context = torch.matmul(weights, values)
        output = self.output_projection(self.merge_heads(context))
        return output, weights


class TransformerEncoderBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.norm1 = nn.LayerNorm(config.embedding_dim)
        self.attention = ExplicitMultiHeadAttention(
            config.embedding_dim,
            config.heads,
        )
        self.norm2 = nn.LayerNorm(config.embedding_dim)
        self.mlp = nn.Sequential(
            nn.Linear(config.embedding_dim, config.mlp_dim),
            nn.GELU(),
            nn.Linear(config.mlp_dim, config.embedding_dim),
        )

    def forward(self, tokens):
        normalized = self.norm1(tokens)
        attended, _ = self.attention(
            normalized,
            normalized,
            normalized,
        )
        tokens = tokens + attended
        return tokens + self.mlp(self.norm2(tokens))


class VisionTransformerMini(nn.Module):
    def __init__(self, config):
        super().__init__()
        patch_dim = config.patch_size * config.patch_size
        self.config = config
        self.patch_projection = nn.Linear(patch_dim, config.embedding_dim)
        self.class_token = nn.Parameter(torch.zeros(1, 1, config.embedding_dim))
        self.position = nn.Parameter(
            torch.zeros(1, config.patch_count + 1, config.embedding_dim)
        )
        self.encoder = TransformerEncoderBlock(config)
        self.norm = nn.LayerNorm(config.embedding_dim)
        self.classifier = nn.Linear(config.embedding_dim, config.num_classes)

    def embed(self, inputs):
        patches = patchify(inputs, self.config.patch_size)
        tokens = self.patch_projection(patches)
        class_tokens = self.class_token.expand(len(inputs), -1, -1)
        return torch.cat((class_tokens, tokens), dim=1) + self.position

    def forward(self, inputs):
        encoded = self.encoder(self.embed(inputs))
        return self.classifier(self.norm(encoded[:, 0]))


model = VisionTransformerMini(config).to(DEVICE)
embedded = model.embed(images[:2])
normalized = model.encoder.norm1(embedded)
_, attention_weights = model.encoder.attention(
    normalized,
    normalized,
    normalized,
)
assert embedded.shape == (2, 17, 24)
assert attention_weights.shape == (2, 4, 17, 17)
assert torch.allclose(
    attention_weights.sum(dim=-1),
    torch.ones_like(attention_weights[..., 0]),
)
assert model(images[:2]).shape == (2, 4)

### 구현 단계 3 — 원문 근거를 코드로 옮기기

- **논문의 어느 부분인가:** §3.2: fine-tuning
- **노트북에서 구현할 부분:** Task 3
- **구현 이유:** 논문의 계산 경로를 작은 tensor로 보존하면서 각 중간값을
  assertion으로 관찰하기 위해서다.
- **완료 증거:** loss/accuracy

먼저 입력 shape와 기대 출력 shape를 종이에 적은 뒤 아래 TODO를 구현한다.

In [ ]:
def classification_loss(logits, targets):
    return F.cross_entropy(logits, targets)


def train_step(model, optimizer, inputs, targets):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    logits = model(inputs)
    loss = classification_loss(logits, targets)
    loss.backward()
    optimizer.step()
    return float(loss.detach())


def evaluate_classifier(model, inputs, targets):
    model.eval()
    with torch.no_grad():
        predictions = model(inputs).argmax(dim=1)
    return float((predictions == targets).float().mean())


optimizer = torch.optim.Adam(model.parameters(), lr=config.learning_rate)
losses = []
for step in range(config.steps):
    step_loss = train_step(model, optimizer, x_train, y_train)
    losses.append(step_loss)

accuracy = evaluate_classifier(
    model,
    images[test_idx],
    labels[test_idx],
)
assert min(losses[3:]) < losses[0]
assert 0.0 <= accuracy <= 1.0
plt.figure(figsize=(5, 3))
plt.plot(losses)
plt.xlabel("step")
plt.ylabel("cross entropy")
plt.title("Vision Transformer mini: accuracy={:.1%}".format(accuracy))
plt.tight_layout()
plt.show()
print(f"loss={losses[0]:.3f} -> {losses[-1]:.3f}")

## 정답 해석 가이드

핵심은 최종 숫자를 외우는 것이 아니라 **원문 위치 → 수식 → 코드 → 반증 가능한 증거**를
연결하는 것입니다. 이 노트북은 다음 원 규모 조건을 재현하지 않았습니다:

> 224×224/patch16, 대규모 사전학습 대신 16×16/patch4와 1 encoder layer를 쓴다.

다음 실험에서는 데이터 크기, 모델 폭, 학습 step 중 하나만 바꾸고 동일 seed에서 비교하세요.